# ДЗ 5. Первый взгляд на orders.csv в Pandas
Срок: **до воскресенья 22.11, 24:00**.
Сдача: папка `hw05` репозитория `dsa-homework` (`hw05.ipynb`): `git add hw05` → `git commit` → `git push`, ссылку прикрепите в LMS.

Каждый ответ: **число и вывод для менеджера** (в print или Markdown-ячейке). Менеджер не читает код: вывод отвечает, что видно в данных и что делать.

Инструменты: Pandas с урока 5 и всё, что было раньше. `groupby`, `apply`, `.str`-методы (кроме готовой строки для `amount`) пока не используем. Если берёте непройденный инструмент, объясните его в комментарии.

- **[База]** обязательно для всех · **[Стандарт]** сколько получится · **[Челлендж]** для тех, кому легко.

## [Для всех] Чтение кода: без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
import pandas as pd
# 1
s = pd.Series([10, 20, 30], index=["a", "b", "c"])
print(s["b"], s.iloc[0])
# 2
a = pd.Series({"Алматы": 5, "Шымкент": 2})
b = pd.Series({"Шымкент": 1, "Астана": 4})
print(a + b)
# 3
df = pd.DataFrame({"x": [1, 2, 3], "y": [4, 5, 6]})
print(df[df["x"] > 1].shape)
# 4
print(type(df["x"]).__name__, type(df[["x"]]).__name__)
# 5
f = df[df["x"] > 1]
print(f.index.tolist(), f.iloc[0, 1])
```

Ваши ответы и объяснения:

1. 
2. 
3. 
4. 
5. 

In [13]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

orders.csv на месте: True


In [14]:
import pandas as pd
import numpy as np

df = pd.read_csv("orders.csv")
# Готовая строка: суммы вида "15 990" записаны текстом. Убираем пробел и переводим в числа (подробно на уроке 6)
df["amount"] = pd.to_numeric(df["amount"].str.replace(" ", ""))
print(df.shape)

(5463, 8)


In [15]:
# Даты записаны текстом "2026-09-01 08:00", поэтому строки можно сравнивать
sep = df["order_date"] < "2026-10"     # сентябрь
octo = ~sep                            # октябрь
paid = df["status"] == "оплачен"
not_test = df["customer_id"] != "TEST"
SH = ["Шымкент", "шымкент", "Shymkent"]
print("сентябрь:", sep.sum(), "| октябрь:", octo.sum())

сентябрь: 2782 | октябрь: 2681


## [База] 1. Паспорт таблицы
Сколько строк и столбцов? Какие типы у столбцов? Сколько пропусков в `delivery_days`? Сколько разных значений в `city`, `category`, `status`, `customer_id`?
Напишите менеджеру три проблемы данных, которые видно уже сейчас, и чем каждая опасна для отчёта о выручке.

In [16]:
# 1
print(df.shape)
print(df.dtypes)
print("Пропуски: ", df['amount'].isna().sum())
print()
for col in ['city', 'category', 'status', 'customer_id']:
    print(df[col].value_counts())

print("1. Одинаковые города не складываются из за того что названия пишутся по разному, 2. Количество заказов не отфильтровано(там есть и оплаченные и неоплаченные)")

(5463, 8)
order_id           int64
order_date           str
customer_id          str
city                 str
category             str
amount           float64
delivery_days    float64
status               str
dtype: object
Пропуски:  0

city
Алматы       1927
Астана       1226
Шымкент       711
Караганда     538
Актобе        397
Атырау        146
астана         84
Astana         80
Almaty         69
 Алматы        66
алматы         64
АЛМАТЫ         62
шымкент        54
Shymkent       39
Name: count, dtype: int64
category
Одежда         1529
Электроника    1185
Дом            1105
Красота         991
Книги           653
Name: count, dtype: int64
status
оплачен    4904
отменён     559
Name: count, dtype: int64
customer_id
C01818    11
C01076     9
C01947     9
C01656     9
C00420     9
          ..
C00645     1
C01442     1
C00018     1
C01650     1
C01057     1
Name: count, Length: 2003, dtype: int64
1. Одинаковые города не складываются из за того что названия пишутся по разному, 2. 

## [База] 2. Типичный заказ
Для оплаченных заказов без TEST: среднее, медиана, 25-й и 75-й перцентили суммы.
Какой чек «типичный»? Что честнее показать в отчёте, среднее или медиану, и почему?

In [47]:
# 2
df_paid = df[(df['status'] == 'оплачен') & (df['customer_id'] != "TEST")]
print('mean:', df_paid['amount'].mean())
print('median:', df_paid['amount'].median())
print(df_paid['amount'].quantile([0.25, 0.75]))

mean: 14672.391348704346
median: 10300.0
0.25     5880.0
0.75    18690.0
Name: amount, dtype: float64


In [17]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5463 entries, 0 to 5462
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   order_id       5463 non-null   int64  
 1   order_date     5463 non-null   str    
 2   customer_id    5463 non-null   str    
 3   city           5463 non-null   str    
 4   category       5463 non-null   str    
 5   amount         5463 non-null   float64
 6   delivery_days  5303 non-null   float64
 7   status         5463 non-null   str    
dtypes: float64(2), int64(1), str(5)
memory usage: 671.2 KB


## [База] 3. Отмены по городам
Доля отменённых заказов: по всем заказам, в Шымкенте, в Алматы (все написания каждого города через `isin`).
Какой город выделяется и во сколько раз?

In [46]:
# 3
ATY = ["Almaty", "almaty", "Алматы", "алматы", "АЛМАТЫ"]
SH = ["Шымкент", "шымкент", "Shymkent"]
a = len(df[unpaid & df["city"].isin(SH)])/df["city"].isin(SH).sum()
b = len(df[unpaid & df['city'].isin(ATY)])/df['city'].isin(ATY).sum()
c = len(df[unpaid])/len(df['status'])
unpaid = df['status'] == 'отменён'
print(f"{c:.1%}")
print(f"{a:.1%}")
print(f"{b:.1%}")
print(f"Шымкент выделяется так как его доля больше в {a/b:.1f} чем доля Алматы и в {a/c:.1f} больше чем по всем заказам")


10.2%
20.4%
8.0%
Шымкент выделяется так как его доля больше в 2.6 чем доля Алматы и в 2.0 больше чем по всем заказам


## [Стандарт] 4. Октябрь против сентября по двум городам
Выручка оплаченных заказов без TEST за сентябрь и октябрь в Шымкенте и в Алматы, изменение в %.
Дубли пока не убираем. Где падение сильнее и во сколько раз? Что сказать менеджеру?

In [60]:
# 4

sep = df_paid["order_date"] < "2026-10"
october = ~sep
shym_sep = df_paid.loc[sep & df_paid['city'].isin(SH), "amount"].sum()
shym_oct = df_paid.loc[october & df_paid['city'].isin(SH), "amount"].sum()
aty_sep = df_paid.loc[sep & df_paid['city'].isin(ATY), 'amount'].sum()
aty_oct = df_paid.loc[october & df_paid['city'].isin(ATY), "amount"].sum()
shym_drop = abs((shym_oct - shym_sep) / shym_sep)
aty_drop = abs((aty_oct - aty_sep) / aty_sep)
print(f'Шымкент: {(shym_oct - shym_sep)/shym_sep:.1%}')
print(f'Алмата: {(aty_oct - aty_sep)/aty_sep:.1%}')
print(f"Падение в Шымкенте сильнее в {shym_drop / aty_drop:.1f} раза")
print("Стоит проверить причину отмен заказов в Шымкенте")

Шымкент: -18.3%
Алмата: -5.1%
Падение в Шымкенте сильнее в 3.6 раза


## [Стандарт] 5. Исправьте код от AI
AI написал код «отменённые заказы Шымкента». В нём **две** ошибки: одна роняет код, вторая тихо даёт неверное число. Найдите обе и исправьте.
```python
cancel_sh = df[df["city"] == "Шымкент" & df["status"] == "отменён"]
print(len(cancel_sh))
```

In [63]:
# 5
cancel_sh = df[(df["city"].isin(SH)) & (df["status"] == "отменён")]
print(len(cancel_sh))


164


## [Стандарт] 6. Самые частые клиенты
Кто сделал больше всего заказов (`value_counts` по `customer_id`)? Покажите все заказы этого клиента (`loc` с маской, столбцы дата, город, сумма, статус).
Есть ли среди его заказов повторяющиеся `order_id`? Что это значит для выручки, которую мы считали весь урок?

In [87]:
# 6
c = df["customer_id"].value_counts().idxmax()
print(f"{c} : {df["customer_id"].value_counts().max()}")
c1 = df.loc[df['customer_id'] == c, ['order_date', 'city', 'amount', 'status']]
print(c1)
print(f'Есть {(df["customer_id"].value_counts().max()) - (df.loc[df['customer_id'] == c, 'order_id'].nunique())} повторяющийся заказ, поэтому выручка завышается')

C01818 : 11
            order_date       city   amount   status
57    2026-09-01 15:00    Шымкент   7350.0  оплачен
779   2026-09-09 18:00    Шымкент  45270.0  оплачен
1371  2026-09-15 21:00     Алматы   8890.0  оплачен
1375  2026-09-15 21:00     Алматы   8890.0  оплачен
2195  2026-09-24 18:00     Астана   1490.0  оплачен
2617  2026-09-29 09:00     Астана  22170.0  оплачен
2776  2026-09-30 21:00     Алматы  20180.0  оплачен
3417  2026-10-08 14:00  Караганда   4330.0  оплачен
4279  2026-10-18 08:00     Астана  13600.0  оплачен
4610  2026-10-21 19:00     Алматы  21990.0  оплачен
4778  2026-10-23 18:00     Алматы  11140.0  оплачен
Есть 1 повторяющийся заказ, поэтому выручка завышается


In [71]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5463 entries, 0 to 5462
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   order_id       5463 non-null   int64  
 1   order_date     5463 non-null   str    
 2   customer_id    5463 non-null   str    
 3   city           5463 non-null   str    
 4   category       5463 non-null   str    
 5   amount         5463 non-null   float64
 6   delivery_days  5303 non-null   float64
 7   status         5463 non-null   str    
dtypes: float64(2), int64(1), str(5)
memory usage: 671.2 KB


## [Челлендж] 7. Выбросы по правилу IQR
Для оплаченных заказов без TEST: `IQR = Q3 − Q1`, граница = `Q3 + 1.5 * IQR`. Сколько заказов выше границы, какая это доля и какая у них категория чаще всего?
Сравните с ДЗ 4: по z-оценке выбросов было 87. Почему правила дают разное число?

In [11]:
# 7

## [Челлендж] 8. Доля отмен для каждого написания города одной строкой
Поделите две Series: число отменённых заказов по `city` и число всех заказов по `city` (обе через `value_counts`). Отсортируйте.
Pandas делит по ярлыкам: проверьте это. Доля отмен у «Шымкент», «шымкент» и «Shymkent» сильно разная. Сколько заказов у каждого написания?
Можно ли судить о городе по одному написанию? Что нужно сделать с городами до анализа (это тема урока 6)?

In [12]:
# 8